In [1]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
import random

random.seed(1234)

PROMPT = '你是一个医学专家，你需要根据用户提出的问题，给出带有思考的回答。'

MAX_LENGTH = 2048

In [4]:
import json

def split_dataset(data_root_path, train_set_path, eval_set_path):
    """
    数据集划分函数，将原始数据集划分为训练集和验证集

    Args:
        data_root_path (str): 原始数据集的根目录路径
        train_set_path (str): 训练集的保存路径
        eval_set_path (str): 验证集的保存路径
    """

    # 5.1初始化数据存放的列表
    data_list = []

    # 5.2 读取原始数据集文件
    with open(data_root_path, 'r', encoding='utf-8') as f:

        # 开始逐行读取
        for line in f:

            # 每一行都先去掉空格
            if line.strip():
                try: # 很可能加载一行数据的时候会出问题，所以异常处理一下
                    # 将读取到的每一行
                    data_list.append(json.loads(line))
                except json.JSONDecodeError:

                    # 打印解析失败的行
                    print(f"Error decoding JSON - 解析失败的行: {line}")

    # 5.3 等待open读取完毕，data_list就包含读取到的所有IFT指令数据，现在对其增加随机性，随机重排
    random.shuffle(data_list)
    print(f'测试 - 加载的数据集大小：已成功加载:{len(data_list)}条数据！')

    # 5.4 开始划分数据集，比例按照9:1划分吧
    split_idx = int(len(data_list) * 0.9)

    # 5.5 按照该比例开始划分数据
    train_data = data_list[:split_idx]
    eval_data = data_list[split_idx:]

    # 5.6 将训练集保存为JSONL格式
    def save_jsonl(datas, file_paths):
        for data, file_path in zip(datas, file_paths):
            with open(file_path, 'w', encoding='utf-8') as f:
                for item in data:
                    json.dump(item, f, ensure_ascii=False)
                    f.write('\n')
    save_jsonl(datas=[train_data, eval_data], file_paths=[train_set_path, eval_set_path])

    print('数据划分部分成功执行完毕！数据集已成功划分！')
    print(f'训练集大小：{len(train_data)}')
    print(f'验证集大小：{len(eval_data)}')

if not os.path.exists('./data/train.jsonl'):
    os.makedirs('./data')
    split_dataset(
        data_root_path='../../source/datasets/delicate_medical_r1_data/r1_data_example.jsonl',
        train_set_path='./data/train.jsonl',
        eval_set_path='./data/eval.jsonl'
    )

测试 - 加载的数据集大小：已成功加载:2407条数据！
数据划分部分成功执行完毕！数据集已成功划分！
训练集大小：2166
验证集大小：241


In [5]:
def dataset_jsonl_transfer(root_path, new_path):
    """
    数据预处理第一部分函数，将原始的jsonl格式数据，转换为Qwen3-1.7B模型所需的指令微调格式

    Args:
        root_path (str): 原始数据集的根目录路径
        new_path (str): 处理完毕后的数据集的保存路径
    """

    # 6.1 准备数据构造的空数组
    messages = []

    # 6.2 读取原始样本子集「划分完毕后的训练集或验证集」数据
    with open(root_path, 'r', encoding='utf-8') as f:

        for line in f:

            # 解析一行数据
            data = json.loads(line)

            # 从中提取出question，提取问题
            input = data['question']

            # 再提取出输出格式信息，包含同时提取thinking思考过程和answer回答过程
            # 所以，其实单纯提取thinking或answer是很简单的，但微调本身要做的就是赋予模型以指令遵循的能力
            # 所以这里我们就必须在微调数据层面就要构造好带有深度思考的问答模型其在输出答案的时候，要遵循的最基本指令格式
            output = f"<think>{data['think']}</think> \n\n {data['answer']}"

            # 构建指令格式的消息「message」
            message = {
                'instruction': PROMPT, # 系统提示词
                'input': f"{input}", # 用户输入
                'output': output # 模型的输出
            }

            # 添加到总的messages数组
            messages.append(message)

    # 6.3 保存重构后的JSONL文件
    with open(new_path, 'w', encoding='utf-8') as f:
        for message in messages:
            f.write(json.dumps(message, ensure_ascii=False) + '\n')

train_dataset_path = './data/train.jsonl'
eval_dataset_path = './data/eval.jsonl'

train_jsonl_new_path = './data/train_format.jsonl'
eval_jsonl_new_path = './data/eval_format.jsonl'

if not os.path.exists(train_jsonl_new_path):
    dataset_jsonl_transfer(root_path=train_dataset_path, new_path=train_jsonl_new_path)
if not os.path.exists(eval_jsonl_new_path):
    dataset_jsonl_transfer(root_path=eval_dataset_path, new_path=eval_jsonl_new_path)

In [8]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq

tokenizer = AutoTokenizer.from_pretrained('../../source/lesson_models/Qwen3-1.7B', use_fast=False, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained('../../source/lesson_models/Qwen3-1.7B', device_map='auto', dtype=torch.bfloat16)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [9]:
model.enable_input_require_grads()

In [10]:
import pandas as pd

def process_data_tokenizer(example):
    """
    数据预处理第二部分函数，使用加载完毕的Tokenizer分词器对输入数据做最后的处理，使其构造为符合模型在微调训练时所需的输入格式

    Args:
        example (dict): 输入的样本数据，包含'instruction', 'input', 'output'等键值对
    """

    # 8.1 先初始化三个列表用于存储处理后的数据
    input_ids, attention_mask, labels = [], [], []

    # 8.2 使用特定的对话格式对instruction部分先进行编码
    instruction = tokenizer(f"<|im_start|>system\n \
              {example['instruction']} \
                <|im_end|>\n \
                    <|im_start|>user\n \
                        {example['input']} \
                            <|im_end|>\n \
                                <|im_start|>assistant\n", add_special_tokens=False)
    # 8.3 再对模型的输出response部分进行编码
    response = tokenizer(f"{example['output']}", add_special_tokens=False)

    # 8.4 拼接instruction和response的token ids，并在末尾添加pad token
    input_ids = instruction['input_ids'] + response['input_ids'] + [tokenizer.pad_token_id]

    # 8.5 拼接attention_mask
    attention_mask = (instruction['attention_mask'] + response['attention_mask'] + [1])

    # 8.6 拼接labels - 在拼接的labels中，instruction部分使用-100表示「含义是不计算损失」，response部分则保留原值
    labels = [-100] * len(instruction['input_ids']) + response['input_ids'] + [tokenizer.pad_token_id]

    # 8.7 如果构造的序列超过了最大长度，则直接截断
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]

    # 最终返回构造好的input_ids,attention_mask,labels
    return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}

train_df = pd.read_json(train_jsonl_new_path, lines=True)
train_ds = Dataset.from_pandas(train_df)
train_dataset = train_ds.map(process_data_tokenizer, remove_columns=train_ds.column_names)

eval_df = pd.read_json(eval_jsonl_new_path, lines=True)
eval_ds = Dataset.from_pandas(eval_df)
eval_dataset = eval_ds.map(process_data_tokenizer, remove_columns=eval_ds.column_names)

train_dataset

Map:   0%|          | 0/2166 [00:00<?, ? examples/s]

Map:   0%|          | 0/241 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2166
})

In [15]:
print(train_dataset[0])

{'input_ids': [151644, 8948, 198, 2290, 220, 56568, 101909, 104316, 101057, 3837, 112735, 100345, 20002, 101080, 103936, 3837, 107485, 106646, 104107, 111423, 1773, 1698, 151645, 198, 2549, 151644, 872, 198, 5108, 96155, 245, 31914, 102388, 9370, 106587, 100166, 104625, 42140, 118901, 63314, 111676, 108165, 100369, 94443, 69041, 111331, 9370, 51255, 34718, 99371, 104384, 3837, 100001, 51255, 34718, 99371, 67338, 63314, 31843, 40820, 103437, 60949, 64064, 101894, 108045, 14009, 51255, 101508, 99762, 527, 100166, 1773, 109194, 100137, 100166, 107126, 31914, 9370, 100407, 105178, 98380, 109916, 100398, 99564, 11319, 4597, 151645, 198, 6656, 151644, 77091, 198, 151667, 106287, 3837, 20002, 56007, 100146, 116473, 102388, 9370, 51255, 101508, 99762, 100166, 107126, 31914, 100407, 105178, 98380, 104126, 1773, 101140, 3837, 35946, 85106, 104843, 100158, 116473, 9370, 100166, 1773, 116473, 112452, 56, 82699, 9370, 3837, 67071, 29258, 63314, 33108, 99578, 63314, 101286, 3837, 68536, 106587, 1001

In [20]:
args = TrainingArguments(
    output_dir="./output/Qwen3-1.7B", # 模型保存路径
    per_device_train_batch_size=1, # 每个设备的训练batch大小
    per_device_eval_batch_size=1, # 每个设备的验证batch大小
    gradient_accumulation_steps=4, # 梯度累积的步数
    eval_strategy='steps', # 评估策略，按训练步进行评估
    eval_steps=100, # 验证集的验证步数
    logging_steps=10, # 训练日志的打印步数
    num_train_epochs=2, # 训练的轮数
    save_steps=400, # 模型保存的步数
    save_total_limit=2, # 最多保存的模型数量
    learning_rate=1e-4, # 学习率
    gradient_checkpointing=True # 开启梯度检查点
)

In [21]:
trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)
    )

The model is already on multiple devices. Skipping the move to device specified in `args`.


In [ ]:
if not os.listdir('./output/Qwen3-1.7B'):
    # 还没训练呢，先训练
    print(f'开始训练！')
    trainer.train()
    print(f'训练完成！即将开始推理测试...')
else:
    # 已经训练过了，直接加载模型
    print(f'已经训练过了，直接加载模型开始推理测试...')

In [ ]:
def predict(messages, model, tokenizer):
    """
    11.1 最终封装一次完整的decoder-only架构的迭代decoder解码过程，即一次推理过程

    Args:
        messages (list): 输入的消息列表，每个消息是一个字典，包含'instruction', 'input', 'output'等键值对
        model (torch.nn.Module): 加载的模型
        tokenizer (transformers.AutoTokenizer): 加载的Tokenizer分词器
    """

    # 11.1 先指定好设备类型
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # 11.2 将对话消息转换为模型的输入格式
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # 11.3 对输入文本进行编码
    model_inputs = tokenizer([text], return_tensors='pt').to(device)

    # 11.4 开始生成回复
    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=MAX_LENGTH,
    )

    # 11.5 从最终的输出内容中提取出模型生成的回复
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]

    # 11.6 对模型生成的回复进行解码
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

checkpoint_root_path = './output/Qwen3-1.7B'

best_checkpoint_path = './output/Qwen3-1.7B/checkpoint-{}'.format(
        max([int(path.split('-')[-1]) for path in os.listdir(checkpoint_root_path)])
    )

chat_tokenizer = AutoTokenizer.from_pretrained(best_checkpoint_path, use_fast=False, trust_remote_code=True)
chat_model = AutoModelForCausalLM.from_pretrained(best_checkpoint_path,device_map='auto', torch_dtype=torch.bfloat16)

test_jsonl_new_path = './data/eval_format.jsonl'
test_df = pd.read_json(test_jsonl_new_path, lines=True)[:3]

test_text_list = []

for index, row in test_df.iterrows():
    instruction = row['instruction']
    input_value = row['input']

    # 构造对话消息
    messages = [
        {
            "role": "system",
            "content": f"{instruction}"
        },
        {
            "role": "user",
            "content": f"{input_value}"
        }
    ]

    # 调用predict进行推理
    response = predict(messages, chat_model, chat_tokenizer)

    # 格式化最终的输出结果
    response_text = f"""
                    Question:{input_value}

                    LLM Answer:{response}
                    """

    print(response_text)